In [ ]:
import os
import time
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from collections import deque
from ultralytics import YOLO
import pandas as pd
import csv

# ------------------------------------------------------------
# ⚙️ CONFIGURATION
# ------------------------------------------------------------
VIDEO_PATH = "/home1/jtt_1/mtp/okutama-action/TestSetVideos/Drone2/Noon/2.2.1.mp4"
VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
OUTPUT_DIR = f"/home1/jtt_1/mtp/outputs-with-tracking-single/{VIDEO_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG = {
    "VIDEO_PATH": VIDEO_PATH,
    "VIDEO_NAME": VIDEO_NAME,
    "OUTPUT_DIR": OUTPUT_DIR,
    
    "YOLO_MODEL_PATH": "/home1/jtt_1/mtp/models-2/yolo-person-model.pt",
    "TSF_MODEL_PATH": "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400",
    "TSF_CHECKPOINT": "/home1/jtt_1/mtp/models-2/best_timesformer_epoch68.pt",

    "TRAIN_CSV": "/home1/jtt_1/mtp/timesformer-person-dataset/train_person_labels.csv",
    "CONF_THRESH": 0.7,
    "DETECT_EVERY_N_FRAMES": 2,
    "OUT_OF_FRAME_FRACTION": 0.4,
    "REID_MEMORY_TIME": 4.0,
    "NUM_FRAMES": 8,
    "TOP_K": 2,
    "TEST_MODE": True,
    "TEST_MAX_FRAMES": 300,
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "RESULTS_CSV": f"{OUTPUT_DIR}/{VIDEO_NAME}_tracking.csv",
}

print(f"[INFO] Processing video: {CONFIG['VIDEO_NAME']}")
print(f"[INFO] Using device: {CONFIG['DEVICE']}")
print(f"[INFO] Output folder: {CONFIG['OUTPUT_DIR']}")

[INFO] Processing video: 2.2.1
[INFO] Using device: cuda
[INFO] Output folder: /home1/jtt_1/mtp/outputs-with-tracking-single/2.2.1


In [2]:
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])

if not cap.isOpened():
    print(f"[ERROR] Could not open video file: {CONFIG['VIDEO_PATH']}")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0

    print(f"[INFO] Video Name       : {CONFIG['VIDEO_NAME']}")
    print(f"[INFO] Total Frames     : {total_frames}")
    print(f"[INFO] FPS              : {fps:.2f}")
    print(f"[INFO] Duration (secs)  : {duration_sec:.2f}")
    print(f"[INFO] Duration (mins)  : {duration_sec/60:.2f}")

cap.release()

[INFO] Video Name       : 2.2.1
[INFO] Total Frames     : 1377
[INFO] FPS              : 29.97
[INFO] Duration (secs)  : 45.95
[INFO] Duration (mins)  : 0.77


In [4]:
# ------------------------------------------------------------
# 🧠 LOAD MODELS
# ------------------------------------------------------------
print("[INFO] Loading YOLO model...")
yolo = YOLO(CONFIG["YOLO_MODEL_PATH"])

print("[INFO] Loading TimeSformer model + labels...")
from transformers import AutoImageProcessor, TimesformerForVideoClassification
from sklearn.preprocessing import MultiLabelBinarizer

train_df = pd.read_csv(CONFIG["TRAIN_CSV"])
all_actions = sorted(list(set(
    sum([str(a).replace("[", "").replace("]", "").replace("'", "").split(",") for a in train_df["Actions"]], [])
)))
label_encoder = MultiLabelBinarizer()
label_encoder.fit([all_actions])
action_labels = label_encoder.classes_

processor = AutoImageProcessor.from_pretrained(CONFIG["TSF_MODEL_PATH"])
tsf_model = TimesformerForVideoClassification.from_pretrained(CONFIG["TSF_MODEL_PATH"])
tsf_model.classifier = torch.nn.Linear(tsf_model.classifier.in_features, len(action_labels))
tsf_model.load_state_dict(torch.load(CONFIG["TSF_CHECKPOINT"], map_location=CONFIG["DEVICE"]))
tsf_model.to(CONFIG["DEVICE"]).eval()
print(f"[INFO] Loaded TimeSformer model with {len(action_labels)} classes")

[INFO] Loading YOLO model...
[INFO] Loading TimeSformer model + labels...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] Loaded TimeSformer model with 13 classes


In [5]:
# ------------------------------------------------------------
# 🎥 VIDEO SETUP
# ------------------------------------------------------------
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if CONFIG["TEST_MODE"]:
    total_frames = min(CONFIG["TEST_MAX_FRAMES"], total_frames)
    print(f"[TEST MODE] Limiting to {total_frames} frames")

print(f"[INFO] FPS={fps:.2f}, size={width}x{height}, total_frames={total_frames}")

[TEST MODE] Limiting to 300 frames
[INFO] FPS=29.97, size=3840x2160, total_frames=300


In [6]:
import numpy as np
from scipy.optimize import linear_sum_assignment
import cv2
import time

# ----------------------
# Helper functions
# ----------------------
def bbox_to_xywh(b):
    x1, y1, x2, y2 = b
    return [(x1 + x2) / 2.0, (y1 + y2) / 2.0, x2 - x1, y2 - y1]

def clamp(v, a, b):
    return max(a, min(b, v))

def crop_box(image, bbox, pad=0):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = int(bbox[0]) - pad, int(bbox[1]) - pad, int(bbox[2]) + pad, int(bbox[3]) + pad
    x1, y1 = clamp(x1, 0, w-1), clamp(y1, 0, h-1)
    x2, y2 = clamp(x2, 0, w-1), clamp(y2, 0, h-1)
    if x2 <= x1 or y2 <= y1:
        return None
    return image[y1:y2, x1:x2]

def color_histogram_feature(image, bbox, size=(32, 32), bins=(16, 16, 8)):
    """Compute a normalized color histogram (HSV) for the bbox region."""
    crop = crop_box(image, bbox)
    if crop is None or crop.size == 0:
        # fallback: small zero vector
        return np.zeros(bins[0] + bins[1] + bins[2], dtype=np.float32)
    crop = cv2.resize(crop, size, interpolation=cv2.INTER_LINEAR)
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h_bins, s_bins, v_bins = bins
    hist_h = cv2.calcHist([hsv], [0], None, [h_bins], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [s_bins], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [v_bins], [0, 256]).flatten()
    hist = np.concatenate([hist_h, hist_s, hist_v]).astype(np.float32)
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def cosine_sim(a, b):
    if a is None or b is None or a.sum()==0 or b.sum()==0:
        return 0.0
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


# ----------------------
# Particle filter class (unchanged core but exposed for predict/estimate)
# ----------------------
class Particle:
    def __init__(self, bbox, num_particles=50):
        self.num_particles = num_particles
        self.particles = self.initialize_particles(bbox)
        self.weights = np.ones(num_particles) / num_particles

    def initialize_particles(self, bbox):
        x1, y1, x2, y2 = bbox
        w, h = max(1, x2 - x1), max(1, y2 - y1)
        p = np.zeros((self.num_particles, 4))
        # sample around the box corners rather than absolute box for stability
        p[:, 0] = np.random.normal((x1 + x2) / 2.0 - w * 0.5, w * 0.05, self.num_particles)
        p[:, 1] = np.random.normal((y1 + y2) / 2.0 - h * 0.5, h * 0.05, self.num_particles)
        p[:, 2] = np.random.normal((x1 + x2) / 2.0 + w * 0.5, w * 0.05, self.num_particles)
        p[:, 3] = np.random.normal((y1 + y2) / 2.0 + h * 0.5, h * 0.05, self.num_particles)
        return p

    def predict(self, sigma=3.0):
        # smaller sigma for smoother motion assumptions
        self.particles += np.random.normal(0, sigma, self.particles.shape)

    def update(self, det):
        d = np.linalg.norm(self.particles - det, axis=1)
        # gaussian weight on L2 distance
        self.weights = np.exp(-d**2 / (2 * (30.0**2)))
        self.weights += 1e-300
        self.weights /= np.sum(self.weights)
        self.resample()

    def resample(self):
        idx = np.random.choice(self.num_particles, self.num_particles, p=self.weights)
        self.particles = self.particles[idx]
        self.weights = np.ones(self.num_particles) / self.num_particles

    def estimate(self):
        return np.average(self.particles, weights=self.weights, axis=0)


# ----------------------
# Track wrapper
# ----------------------
class ParticleTrack:
    def __init__(self, bbox, tid, feature=None, timestamp=None,
                 num_particles=50, confirm_hits=3, ema_alpha=0.2):
        self.id = tid
        self.pf = Particle(bbox, num_particles=num_particles)
        self.lost = 0
        self.age = 1
        self.hits = 1  # number of times associated
        self.confirm_hits = confirm_hits
        self.confirmed = (self.hits >= self.confirm_hits)
        self.last_box = bbox
        self.last_seen = timestamp if timestamp is not None else time.time()
        # appearance feature (vector) - if available, keep EMA average
        self.feature = feature.copy() if feature is not None else None
        self.ema_alpha = ema_alpha

    def predict(self):
        self.pf.predict()
        est = self.pf.estimate()
        # estimated box may be in particle coordinate form: [x1,y1,x2,y2]
        self.age += 1
        return est

    def update(self, bbox, feature=None, timestamp=None):
        self.pf.update(bbox)
        self.last_box = bbox
        self.last_seen = timestamp if timestamp is not None else time.time()
        self.lost = 0
        self.hits += 1
        if feature is not None:
            if self.feature is None:
                self.feature = feature.copy()
            else:
                # exponential moving average to adapt appearance slowly
                self.feature = (1.0 - self.ema_alpha) * self.feature + self.ema_alpha * feature
        self.confirmed = (self.hits >= self.confirm_hits)


# ----------------------
# Tracker main class
# ----------------------
class ParticleTracker:
    def __init__(self,
                 max_lost=30,
                 iou_weight=0.5,
                 dist_weight=0.2,
                 app_weight=0.3,
                 distance_decay=80.0,
                 min_confirm_hits=3,
                 reid_memory_time=3.0,
                 num_particles=50):
        """
        max_lost: frames after which a track is considered gone
        iou_weight, dist_weight, app_weight: weights for the combined matching cost (sum=1)
        distance_decay: used to convert center distance -> similarity
        min_confirm_hits: hits required to mark a track 'confirmed'
        reid_memory_time: seconds to keep recently lost tracks in memory for re-id
        """
        self.tracks = {}            # active tracks: id -> ParticleTrack
        self.next_id = 0
        self.max_lost = max_lost
        self.iou_w = iou_weight
        self.dist_w = dist_weight
        self.app_w = app_weight
        assert abs(self.iou_w + self.dist_w + self.app_w - 1.0) < 1e-6, "weights must sum to 1"
        self.distance_decay = distance_decay
        self.min_confirm_hits = min_confirm_hits
        self.reid_memory_time = reid_memory_time
        self.memory = {}            # id -> (last_box, feature, last_seen_time)
        self.num_particles = num_particles

    # ----------------------
    # geometry / similarity helpers
    # ----------------------
    @staticmethod
    def iou(b1, b2):
        x1, y1, x2, y2 = max(b1[0], b2[0]), max(b1[1], b2[1]), min(b1[2], b2[2]), min(b1[3], b2[3])
        w, h = max(0, x2 - x1), max(0, y2 - y1)
        inter = w * h
        area1 = max(0, (b1[2]-b1[0])*(b1[3]-b1[1]))
        area2 = max(0, (b2[2]-b2[0])*(b2[3]-b2[1]))
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0.0

    @staticmethod
    def center_distance(b1, b2):
        c1 = ((b1[0]+b1[2])/2.0, (b1[1]+b1[3])/2.0)
        c2 = ((b2[0]+b2[2])/2.0, (b2[1]+b2[3])/2.0)
        return np.linalg.norm(np.array(c1) - np.array(c2))

    def appearance_similarity(self, f1, f2):
        # returns similarity in [0,1]
        if f1 is None or f2 is None:
            return 0.0
        return clamp(cosine_sim(f1, f2), 0.0, 1.0)

    # ----------------------
    # Matching
    # ----------------------
    def build_cost_matrix(self, track_boxes, track_features, detections, det_features):
        n_t = len(track_boxes)
        n_d = len(detections)
        if n_t == 0 or n_d == 0:
            return np.zeros((n_t, n_d), dtype=np.float32)

        cost = np.zeros((n_t, n_d), dtype=np.float32)
        for i, tb in enumerate(track_boxes):
            for j, db in enumerate(detections):
                iou_score = self.iou(tb, db)          # [0,1]
                dist = self.center_distance(tb, db)
                dist_score = np.exp(-dist / (self.distance_decay + 1e-9))  # [0,1]
                app_score = self.appearance_similarity(track_features[i], det_features[j])  # [0,1]
                # combined similarity
                sim = self.iou_w * iou_score + self.dist_w * dist_score + self.app_w * app_score
                # cost for hungarian is 1 - similarity (lower better)
                cost[i, j] = 1.0 - sim
        return cost

    # ----------------------
    # Public update: detections can be list of bboxes or list of dict {'bbox':..,'feature':..}
    # frame is required if detections are raw bboxes and we want to compute appearance features
    # ----------------------
    def update(self, detections, frame=None, timestamp=None):
        """
        detections: list of bboxes [x1,y1,x2,y2] OR list of dicts {'bbox':..., 'feature':...}
        frame: current video frame (BGR numpy array) - used to compute appearance features if needed
        returns: list of tuples (bbox, track_id, confirmed_bool)
        """
        timestamp = timestamp if timestamp is not None else time.time()

        # normalize detections to (bbox, feature) pairs
        det_bboxes = []
        det_features = []
        for det in detections:
            if isinstance(det, dict):
                bbox = det['bbox']
                feature = det.get('feature', None)
            else:
                bbox = det
                feature = None
            det_bboxes.append(bbox)
            if feature is None:
                # compute simple color histogram if frame is available, else None
                if frame is not None:
                    feature = color_histogram_feature(frame, bbox)
                else:
                    feature = None
            det_features.append(feature)

        # If no active tracks, create new tracks for all detections
        if not self.tracks:
            outputs = []
            for j, bbox in enumerate(det_bboxes):
                fid = self.next_id
                tr = ParticleTrack(bbox, fid, feature=det_features[j],
                                   timestamp=timestamp, num_particles=self.num_particles,
                                   confirm_hits=self.min_confirm_hits)
                tr.confirmed = tr.hits >= self.min_confirm_hits
                self.tracks[fid] = tr
                self.next_id += 1
                outputs.append((bbox, fid, tr.confirmed))
            return outputs

        # Prepare track lists
        track_ids = list(self.tracks.keys())
        track_boxes = []
        track_features = []
        # predict all tracks and get predicted boxes
        for tid in track_ids:
            tr = self.tracks[tid]
            pred = tr.predict()
            # ensure pred is in bbox form and numerical
            pred_box = [float(pred[0]), float(pred[1]), float(pred[2]), float(pred[3])]
            track_boxes.append(pred_box)
            track_features.append(tr.feature)

        # compute assignment cost and solve
        cost = self.build_cost_matrix(track_boxes, track_features, det_bboxes, det_features)
        row, col = [], []
        if cost.size > 0:
            row, col = linear_sum_assignment(cost)

        assigned_tracks = set()
        assigned_dets = set()
        outputs = []

        # apply matches with a similarity threshold
        for r, c in zip(row, col):
            cval = cost[r, c]
            sim = 1.0 - cval
            # threshold: require at least moderate similarity to accept assignment
            if sim >= 0.25:   # you can tune this (0.25 -> very permissive)
                tid = track_ids[r]
                self.tracks[tid].update(det_bboxes[c], feature=det_features[c], timestamp=timestamp)
                assigned_tracks.add(tid)
                assigned_dets.add(c)
                outputs.append((det_bboxes[c], tid, self.tracks[tid].confirmed))

        # unmatched detections -> attempt memory (reid) or create new track
        for j, bbox in enumerate(det_bboxes):
            if j in assigned_dets:
                continue
            reused_id = None
            # try to match with memory using spatial + appearance similarity
            for mid, (mbox, mfeat, mtime) in list(self.memory.items()):
                # ensure memory age not exceeded
                if timestamp - mtime > self.reid_memory_time:
                    del self.memory[mid]
                    continue
                dist = self.center_distance(bbox, mbox)
                if dist < 120:   # spatial cue for potential reid
                    app_sim = 0.0
                    if mfeat is not None and det_features[j] is not None:
                        app_sim = cosine_sim(mfeat, det_features[j])
                    if app_sim > 0.35:
                        reused_id = mid
                        del self.memory[mid]
                        break
            if reused_id is not None:
                # re-create track with same id and previous feature
                tr = ParticleTrack(bbox, reused_id, feature=det_features[j],
                                   timestamp=timestamp, num_particles=self.num_particles,
                                   confirm_hits=self.min_confirm_hits)
                tr.confirmed = tr.hits >= self.min_confirm_hits
                self.tracks[reused_id] = tr
                outputs.append((bbox, reused_id, tr.confirmed))
                assigned_dets.add(j)
                continue

            # otherwise create new track
            fid = self.next_id
            tr = ParticleTrack(bbox, fid, feature=det_features[j],
                               timestamp=timestamp, num_particles=self.num_particles,
                               confirm_hits=self.min_confirm_hits)
            tr.confirmed = tr.hits >= self.min_confirm_hits
            self.tracks[fid] = tr
            self.next_id += 1
            outputs.append((bbox, fid, tr.confirmed))
            assigned_dets.add(j)

        # update lost counters for unmatched tracks
        for tid in list(self.tracks.keys()):
            if tid not in assigned_tracks:
                tr = self.tracks[tid]
                tr.lost += 1
                # save last box & feature to memory for potential re-id with timestamp
                if tr.lost > 0:
                    self.memory[tid] = (tr.last_box, tr.feature, tr.last_seen)
                if tr.lost > self.max_lost:
                    # expire track permanently
                    if tid in self.tracks:
                        del self.tracks[tid]

        return outputs

    # small utility to get all current confirmed tracks
    def get_confirmed_tracks(self):
        return [(tr.last_box, tid) for tid, tr in self.tracks.items() if tr.confirmed]


In [7]:
# Cell 3: Video setup, fps and Tracker init
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])
if not cap.isOpened():
    raise RuntimeError(f"Could not open {CONFIG['VIDEO_PATH']}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"[INFO] video fps: {fps:.2f}, size: {width}x{height}, frames: {total_frames}")

# set detect frequency if unset
if CONFIG["DETECT_EVERY_N_FRAMES"] is None:
    CONFIG["DETECT_EVERY_N_FRAMES"] = max(1, int(round(fps)))  # once per second by default

if CONFIG["TEST_MODE"]:
    total_frames = min(CONFIG["TEST_MAX_FRAMES"], total_frames)
    print(f"[INFO] TEST_MODE: limiting to {total_frames} frames")

tracker = ParticleTracker(
    max_lost=30,
    iou_weight=0.5,
    dist_weight=0.2,
    app_weight=0.3,
    distance_decay=80.0,
    min_confirm_hits=3,
    reid_memory_time=CONFIG["REID_MEMORY_TIME"],
    num_particles=50
)
print("[INFO] ParticleTracker initialized")


[INFO] video fps: 29.97, size: 3840x2160, frames: 1377
[INFO] TEST_MODE: limiting to 300 frames
[INFO] ParticleTracker initialized


In [8]:
def bbox_area_intersection_fraction(bbox, frame_w, frame_h):
    """Returns intersection area fraction relative to bbox area."""
    x1, y1, x2, y2 = bbox
    bx1, by1 = max(0, x1), max(0, y1)
    bx2, by2 = min(frame_w, x2), min(frame_h, y2)
    inter_w, inter_h = max(0, bx2 - bx1), max(0, by2 - by1)
    inter_area = inter_w * inter_h
    bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
    return inter_area / bbox_area  # in [0,1]

def yolo_to_bboxes(result):
    try:
        boxes = result[0].boxes.xyxy.cpu().numpy()
    except Exception:
        boxes = result.boxes.xyxy.cpu().numpy()
    bboxes = []
    for b in boxes:
        x1, y1, x2, y2 = map(float, b[:4])
        bboxes.append([x1, y1, x2, y2])
    return bboxes

def fraction_tracks_out_of_frame(tracker, frame_w, frame_h, threshold_inside=0.2):
    confirmed = [tr for tr in tracker.tracks.values() if tr.confirmed]
    if not confirmed:
        return 0.0
    out_count = 0
    for tr in confirmed:
        x1, y1, x2, y2 = tr.last_box
        bx1, by1 = max(0, x1), max(0, y1)
        bx2, by2 = min(frame_w, x2), min(frame_h, y2)
        inter_w, inter_h = max(0, bx2 - bx1), max(0, by2 - by1)
        inter_area = inter_w * inter_h
        bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
        frac = inter_area / bbox_area
        if frac < threshold_inside:
            out_count += 1
    return out_count / len(confirmed)



In [9]:
frame_buffers = {}        # track_id -> deque(frames)
action_predictions = {}   # track_id -> (labels, scores)
results_rows = []

NUM_FRAMES = CONFIG["NUM_FRAMES"]
TOP_K = CONFIG["TOP_K"]
frame_idx = 0
start_time = time.time()
last_detect_frame = -999

print(f"[INFO] DETECT_EVERY_N_FRAMES = {CONFIG['DETECT_EVERY_N_FRAMES']}")
print(f"[INFO] Using TimeSformer ({NUM_FRAMES} frames/clip, TOP_K={TOP_K})")

while frame_idx < total_frames:
    ret, frame = cap.read()
    if not ret:
        break

    do_detect = False
    if (frame_idx - last_detect_frame) >= CONFIG["DETECT_EVERY_N_FRAMES"]:
        do_detect = True
    if len(tracker.tracks) == 0:
        do_detect = True
    frac_out = fraction_tracks_out_of_frame(tracker, width, height, threshold_inside=0.2)
    if frac_out >= CONFIG["OUT_OF_FRAME_FRACTION"]:
        do_detect = True

    timestamp = time.time()
    detections = []

    # --- YOLO DETECTION ---
    if do_detect:
        yolo_res = yolo.predict(frame, conf=CONFIG["CONF_THRESH"], verbose=False)
        detections = yolo_to_bboxes(yolo_res)
        last_detect_frame = frame_idx
        updates = tracker.update(detections, frame=frame, timestamp=timestamp)
    else:
        updates = tracker.update([], frame=frame, timestamp=timestamp)

    # --- CLEAN LOST TRACKS ---
    for tid in list(tracker.tracks.keys()):
        tr = tracker.tracks[tid]
        if tr.lost > tracker.max_lost:
            if tid in frame_buffers:
                del frame_buffers[tid]
            if tid in action_predictions:
                del action_predictions[tid]

    # --- TRACKING & ACTIONS ---
    for tid, tr in list(tracker.tracks.items()):
        if not tr.confirmed:
            continue

        x1, y1, x2, y2 = map(int, tr.last_box)
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(width, x2), min(height, y2)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        # store cropped frame in buffer
        if tid not in frame_buffers:
            frame_buffers[tid] = deque(maxlen=NUM_FRAMES)
        frame_buffers[tid].append(cv2.resize(crop, (224, 224)))

        actions, scores = [], []
        # classify when enough frames collected
        if len(frame_buffers[tid]) == NUM_FRAMES and (frame_idx % NUM_FRAMES == 0):
            clip = torch.tensor(np.stack(frame_buffers[tid])).permute(0, 3, 1, 2).unsqueeze(0).float().to(CONFIG["DEVICE"])
            with torch.no_grad():
                logits = tsf_model(clip).logits
                probs = F.softmax(logits, dim=-1)[0]
                top = torch.topk(probs, k=TOP_K)
                actions = [action_labels[i] for i in top.indices.cpu().numpy()]
                scores = [float(p) for p in top.values.cpu().numpy()]
            action_predictions[tid] = (actions, scores)
            print(f"[ACTION] Frame {frame_idx} | Track {tid} -> {actions}")

        if tid in action_predictions:
            actions, scores = action_predictions[tid]
        actions_str = ",".join(actions)
        scores_str = ",".join([f"{s:.2f}" for s in scores])

        results_rows.append((
            frame_idx, tid, float(x1), float(y1), float(x2), float(y2),
            int(tr.confirmed), actions_str, scores_str
        ))

    frame_idx += 1

# ------------------------------------------------------------
# 🧾 SAVE RESULTS
# ------------------------------------------------------------
elapsed = time.time() - start_time
proc_fps = frame_idx / elapsed if elapsed > 0 else 0.0
print(f"[INFO] Processed {frame_idx} frames in {elapsed:.2f}s => {proc_fps:.2f} FPS")
cap.release()

df = pd.DataFrame(results_rows, columns=[
    "frame_idx", "track_id", "x1", "y1", "x2", "y2",
    "confirmed", "actions", "scores"
])
df = df.drop_duplicates(subset=["frame_idx", "track_id"], keep="last")
df.to_csv(CONFIG["RESULTS_CSV"], index=False)

print(f"[INFO] Saved tracking + actions CSV → {CONFIG['RESULTS_CSV']}")

[INFO] DETECT_EVERY_N_FRAMES = 2
[INFO] Using TimeSformer (8 frames/clip, TOP_K=2)


[ACTION] Frame 16 | Track 0 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 16 | Track 1 -> ['Standing', 'Walking']
[ACTION] Frame 24 | Track 0 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 24 | Track 1 -> ['Standing', 'Walking']
[ACTION] Frame 32 | Track 0 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 32 | Track 1 -> ['Standing', 'Walking']
[ACTION] Frame 40 | Track 0 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 56 | Track 0 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 56 | Track 1 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 72 | Track 0 -> ['Standing', 'Walking']
[ACTION] Frame 72 | Track 1 -> ['Standing', 'Pushing or Pulling']
[ACTION] Frame 80 | Track 0 -> ['Standing', 'Walking']
[ACTION] Frame 80 | Track 1 -> ['Standing', 'Walking']
[ACTION] Frame 80 | Track 2 -> ['Standing', 'Walking']
[ACTION] Frame 88 | Track 0 -> ['Standing', 'Walking']
[ACTION] Frame 88 | Track 2 -> ['Standing', 'Walking']
[ACTION] Frame 96 | Track 2 -> ['Standing',

In [10]:
import cv2
import pandas as pd
import os
import random
from tqdm import tqdm
import numpy as np

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
VIDEO_PATH = CONFIG["VIDEO_PATH"]
VIDEO_NAME = CONFIG["VIDEO_NAME"]
OUTPUT_DIR = CONFIG["OUTPUT_DIR"]
CSV_PATH = f"{OUTPUT_DIR}/{VIDEO_NAME}_tracking.csv"
OUTPUT_VIDEO = f"{OUTPUT_DIR}/{VIDEO_NAME}_output.mp4"

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if CONFIG.get("TEST_MODE", False):
    total = min(CONFIG.get("TEST_MAX_FRAMES", 200), total)
print(f"[INFO] Rendering {total} frames with larger readable labels...")

# ------------------------------------------------------------
# DYNAMIC FONT SIZE
# ------------------------------------------------------------
FONT_SCALE = max(0.6, min(1.4, h / 720 * 0.8))
THICKNESS = int(max(2, h / 720 * 2))
PADDING_X, PADDING_Y = int(10 * (h / 720)), int(6 * (h / 720))
print(f"[INFO] Font scale={FONT_SCALE:.2f}, thickness={THICKNESS}")

# ------------------------------------------------------------
# COLOR HELPERS
# ------------------------------------------------------------
track_colors = {}
def get_color(tid):
    if tid not in track_colors:
        random.seed(tid)
        track_colors[tid] = tuple(random.randint(40, 200) for _ in range(3))
    return track_colors[tid]

def lighten_color(color, factor=1.5):
    return tuple(min(255, int(c * factor)) for c in color)

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------
def draw_filled_rounded_rect(img, x1, y1, x2, y2, color, radius=8, alpha=0.95):
    overlay = img.copy()
    cv2.rectangle(overlay, (x1 + radius, y1), (x2 - radius, y2), color, -1)
    cv2.rectangle(overlay, (x1, y1 + radius), (x2, y2 - radius), color, -1)
    cv2.circle(overlay, (x1 + radius, y1 + radius), radius, color, -1)
    cv2.circle(overlay, (x2 - radius, y1 + radius), radius, color, -1)
    cv2.circle(overlay, (x1 + radius, y2 - radius), radius, color, -1)
    cv2.circle(overlay, (x2 - radius, y2 - radius), radius, color, -1)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)

def draw_label(frame, text, pos, color, alpha=0.9):
    """Draw visible text with translucent rounded background."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    x, y = pos
    (tw, th), _ = cv2.getTextSize(text, font, FONT_SCALE, THICKNESS)
    x1, y1 = max(0, x - PADDING_X), max(0, y - th - PADDING_Y)
    x2, y2 = min(frame.shape[1] - 1, x + tw + PADDING_X), min(frame.shape[0] - 1, y + PADDING_Y)

    bg_color = lighten_color(color, 1.5)
    draw_filled_rounded_rect(frame, x1, y1, x2, y2, bg_color, radius=8, alpha=alpha)

    # White text with black outline for contrast
    cv2.putText(frame, text, (x + 3, y - 2), font, FONT_SCALE, (0, 0, 0), THICKNESS + 2, cv2.LINE_AA)
    cv2.putText(frame, text, (x + 3, y - 2), font, FONT_SCALE, (255, 255, 255), THICKNESS, cv2.LINE_AA)

# ------------------------------------------------------------
# RENDER LOOP
# ------------------------------------------------------------
frame_groups = df.groupby("frame_idx")
pbar = tqdm(total=total, desc="[Rendering Video]", unit="frame")

for i in range(total):
    ret, frame = cap.read()
    if not ret:
        break

    if i in frame_groups.groups:
        g = frame_groups.get_group(i)
        for _, r in g.iterrows():
            x1, y1, x2, y2 = int(r.x1), int(r.y1), int(r.x2), int(r.y2)
            tid, actions = int(r.track_id), str(r.actions).strip()
            color = get_color(tid)

            # Skip if actions are NaN, empty, or invalid
            if actions.lower() in ["nan", "none", "null", ""]:
                actions = ""

            # Bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, THICKNESS)

            # Top label (Track ID)
            draw_label(frame, f"ID {tid}", (x1, y1 - 10), color, alpha=0.9)

            # Bottom label (Multi-label actions)
            if actions:
                act_text = " | ".join([a.strip() for a in actions.split(",") if a.strip().lower() not in ["nan", "none", "null"]])
                if act_text:
                    label_y = y2 + int(30 * (h / 720))
                    if label_y + 25 > h:
                        label_y = max(25, y1 - int(40 * (h / 720)))
                    draw_label(frame, act_text, (x1, label_y), color, alpha=0.95)

    out.write(frame)
    pbar.update(1)

pbar.close()
cap.release()
out.release()

print(f"[INFO] Visualization saved → {OUTPUT_VIDEO}")


[INFO] Rendering 300 frames with larger readable labels...
[INFO] Font scale=1.40, thickness=6


[Rendering Video]:   0%|          | 0/300 [00:00<?, ?frame/s]

[Rendering Video]: 100%|██████████| 300/300 [00:22<00:00, 13.46frame/s]

[INFO] Visualization saved → /home1/jtt_1/mtp/outputs-with-tracking-single/2.2.1/2.2.1_output.mp4


In [12]:
# # ============================================================
# # 🚀 MAIN PROCESSING LOOP (with TEST_MODE + YOLO-based prediction)
# # ============================================================

# frame_buffer = {}
# last_predictions = {}
# start_time = time.time()

# print("[INFO] Starting frame processing...")

# # Limit frame count if test mode
# max_frames = 100 if CONFIG.get("TEST_MODE", False) else total_frames
# if CONFIG.get("TEST_MODE", False):
#     print(f"🧪 TEST_MODE=True → Only processing first {max_frames} frames")

# for frame_idx in tqdm(range(max_frames), desc="Frames", ncols=100):
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # -------------------------------
#     # YOLO Detection (person only)
#     # -------------------------------
#     detections = yolo(frame, verbose=False)[0]
#     persons = [
#         box for box in detections.boxes
#         if int(box.cls[0]) == 0 and float(box.conf[0]) >= CONFIG["CONF_THRESH"]
#     ]

#     if not persons:
#         # No persons detected → skip prediction and writing
#         out.write(frame)
#         continue

#     # -------------------------------
#     # Process each detected person
#     # -------------------------------
#     for pid, box in enumerate(persons):
#         x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

#         # Skip invalid crops
#         if x2 <= x1 or y2 <= y1:
#             continue

#         crop = frame[y1:y2, x1:x2]
#         if crop.size == 0:
#             continue

#         # Maintain a frame buffer for each detected ID (pid)
#         if pid not in frame_buffer:
#             frame_buffer[pid] = []

#         resized_crop = cv2.resize(crop, (224, 224))
#         frame_buffer[pid].append(resized_crop)

#         # Keep only the latest N frames per ID
#         if len(frame_buffer[pid]) > CONFIG["NUM_FRAMES"]:
#             frame_buffer[pid].pop(0)

#         # ----------------------------------------
#         # Run Transformer prediction periodically
#         # ----------------------------------------
#         if len(frame_buffer[pid]) == CONFIG["NUM_FRAMES"] and frame_idx % CONFIG["SKIP_FRAMES"] == 0:
#             frames_seq = frame_buffer[pid]

#             inputs = processor(frames_seq, return_tensors="pt")
#             pixel_values = inputs["pixel_values"].squeeze(0).unsqueeze(0).to(CONFIG["DEVICE"])

#             with torch.no_grad():
#                 outputs = tsf_model(pixel_values)
#                 probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

#             # Select top labels above threshold
#             top_idx = np.argsort(probs)[::-1][:CONFIG["TOP_K"]]
#             pred_labels = [
#                 f"{action_labels[i]} ({probs[i]:.2f})"
#                 for i in top_idx if probs[i] > 0.3
#             ]
#             label_text = "Person"
#             label_text = " | ".join(pred_labels) if pred_labels else "None"
#             last_predictions[pid] = label_text
#         else:
#             label_text = "Person"
#             label_text = last_predictions.get(pid, "")

#         # ----------------------------------------
#         # Draw only YOLO-detected bounding boxes
#         # ----------------------------------------
#         # cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

#         # if label_text:
#         #     (tw, th), _ = cv2.getTextSize(label_text, font, 0.6, 2)
#         #     y_text = max(25, y1 - 10)
#         #     overlay = frame.copy()
#         #     cv2.rectangle(overlay, (x1, y_text - th - 8), (x1 + tw + 12, y_text + 6), (0, 0, 0), -1)
#         #     frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)
#         #     cv2.putText(frame, label_text, (x1 + 6, y_text),
#         #                 font, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

#     # -------------------------------
#     # Overlay FPS, RAM, GPU usage, etc.
#     # -------------------------------
#         draw_metrics_on_frame(frame, CONFIG["VIDEO_NAME"], frame_idx + 1, start_time)
#     out.write(frame)

# # ============================================================
# # ✅ Clean-up
# # ============================================================
# cap.release()
# out.release()

# print(f"\n✅ {'TEST' if CONFIG.get('TEST_MODE', False) else 'FULL'} mode completed.")
# print(f"💾 Saved annotated video: {CONFIG['OUTPUT_PATH']}")


In [13]:
# from collections import deque
# import torch
# import torch.nn.functional as F
# import numpy as np
# import pandas as pd
# import cv2, time
# from tqdm import tqdm
# import random, os

# # ------------------------------------------------------------
# # ⚙️ CONFIG
# # ------------------------------------------------------------
# frame_buffers = {}
# action_predictions = {}
# results_rows = []

# NUM_FRAMES = CONFIG["NUM_FRAMES"]
# TOP_K = CONFIG["TOP_K"]
# TSF_CONF_THRESH = CONFIG.get("TSF_CONF_THRESH", 0.1)  # 🔹 confidence threshold for display/logging only
# MAX_ACTIONS = 2  # 🔹 keep top N actions after cleaning
# frame_idx = 0
# start_time = time.time()
# last_detect_frame = -999

# print(f"[INFO] DETECT_EVERY_N_FRAMES = {CONFIG['DETECT_EVERY_N_FRAMES']}")
# print(f"[INFO] Using TimeSformer ({NUM_FRAMES} frames/clip, TOP_K={TOP_K}, CONF_THRESH={TSF_CONF_THRESH}, MAX_ACTIONS={MAX_ACTIONS})")

# # ------------------------------------------------------------
# # 🧱 IMPOSSIBLE ACTION COMBINATIONS
# # ------------------------------------------------------------
# IMPOSSIBLE_COMBOS = [
#     ("Running", "Standing"),
#     ("Running", "Sitting"),
#     ("Running", "Lying"),
#     ("Standing", "Lying"),
#     ("Walking", "Lying"),
#     ("Running", "Reading"),
#     ("Sitting", "Walking"),
#     ("Standing", "Sitting"),
#     ("Lying", "Standing"),
#     ("Handshaking", "Hugging")
# ]
# IMPOSSIBLE_COMBOS = [tuple(sorted(pair)) for pair in IMPOSSIBLE_COMBOS]

# def clean_impossible_combos(actions_list, scores_dict):
#     """Remove impossible combos, keeping higher-confidence one."""
#     cleaned = set(actions_list)
#     changed = False
#     for a1, a2 in IMPOSSIBLE_COMBOS:
#         if a1 in cleaned and a2 in cleaned:
#             s1, s2 = scores_dict.get(a1, 0), scores_dict.get(a2, 0)
#             if s1 >= s2:
#                 cleaned.remove(a2)
#             else:
#                 cleaned.remove(a1)
#             changed = True
#     return list(cleaned), changed


# # ------------------------------------------------------------
# # 🚀 MAIN LOOP
# # ------------------------------------------------------------
# while frame_idx < total_frames:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # YOLO detection trigger logic
#     do_detect = False
#     if (frame_idx - last_detect_frame) >= CONFIG["DETECT_EVERY_N_FRAMES"]:
#         do_detect = True
#     if len(tracker.tracks) == 0:
#         do_detect = True
#     frac_out = fraction_tracks_out_of_frame(tracker, width, height, threshold_inside=0.2)
#     if frac_out >= CONFIG["OUT_OF_FRAME_FRACTION"]:
#         do_detect = True

#     timestamp = time.time()
#     detections = []

#     if do_detect:
#         yolo_res = yolo.predict(frame, conf=CONFIG["CONF_THRESH"], verbose=False)
#         detections = yolo_to_bboxes(yolo_res)
#         last_detect_frame = frame_idx
#         updates = tracker.update(detections, frame=frame, timestamp=timestamp)
#     else:
#         updates = tracker.update([], frame=frame, timestamp=timestamp)

#     # Clean lost tracks
#     for tid in list(tracker.tracks.keys()):
#         tr = tracker.tracks[tid]
#         if tr.lost > tracker.max_lost:
#             frame_buffers.pop(tid, None)
#             action_predictions.pop(tid, None)

#     # --- Track + Predict actions ---
#     for tid, tr in list(tracker.tracks.items()):
#         if not tr.confirmed:
#             continue

#         x1, y1, x2, y2 = map(int, tr.last_box)
#         x1, y1, x2, y2 = max(0, x1), max(0, y1), min(width, x2), min(height, y2)
#         crop = frame[y1:y2, x1:x2]
#         if crop.size == 0:
#             continue

#         # Maintain frame buffer
#         if tid not in frame_buffers:
#             frame_buffers[tid] = deque(maxlen=NUM_FRAMES)
#         frame_buffers[tid].append(cv2.resize(crop, (224, 224)))

#         actions, scores = [], []
#         if len(frame_buffers[tid]) == NUM_FRAMES and (frame_idx % NUM_FRAMES == 0):
#             clip = torch.tensor(np.stack(frame_buffers[tid])).permute(0, 3, 1, 2).unsqueeze(0).float().to(CONFIG["DEVICE"])
#             with torch.no_grad():
#                 logits = tsf_model(clip).logits
#                 probs = torch.sigmoid(logits)[0]  # ✅ multi-label output

#                 # Take *all* predictions, sorted by confidence
#                 sorted_idx = torch.argsort(probs, descending=True)
#                 actions = [action_labels[i] for i in sorted_idx.cpu().numpy()]
#                 scores = [float(probs[i].cpu().item()) for i in sorted_idx.cpu().numpy()]
#                 score_map = dict(zip(actions, scores))

#             # --- Clean impossible combinations from full list ---
#             cleaned_actions, changed = clean_impossible_combos(actions, score_map)
#             if changed:
#                 print(f"[WARN] Frame {frame_idx} | Track {tid}: impossible combo cleaned")

#             # --- Keep top-N actions after cleaning ---
#             top_actions = sorted(cleaned_actions, key=lambda a: score_map[a], reverse=True)[:MAX_ACTIONS]
#             top_scores = [score_map[a] for a in top_actions]

#             # --- Optional filter for confidence ---
#             confident_actions = [a for a, s in zip(top_actions, top_scores) if s >= TSF_CONF_THRESH]
#             confident_scores = [s for s in top_scores if s >= TSF_CONF_THRESH]

#             if confident_actions:
#                 action_predictions[tid] = (confident_actions, confident_scores)
#                 print(f"[ACTION] Frame {frame_idx} | Track {tid} -> {confident_actions} @ {confident_scores}")
#             else:
#                 print(f"[INFO] Frame {frame_idx} | Track {tid}: low-confidence, keeping previous")

#         # Retrieve last predictions
#         if tid in action_predictions:
#             actions, scores = action_predictions[tid]

#         # Save to results
#         actions_str = ",".join(actions)
#         scores_str = ",".join([f"{s:.2f}" for s in scores])
#         results_rows.append((
#             frame_idx, tid, float(x1), float(y1), float(x2), float(y2),
#             int(tr.confirmed), actions_str, scores_str
#         ))

#     frame_idx += 1


# # ------------------------------------------------------------
# # 🧾 SAVE RESULTS
# # ------------------------------------------------------------
# elapsed = time.time() - start_time
# proc_fps = frame_idx / elapsed if elapsed > 0 else 0.0
# print(f"[INFO] Processed {frame_idx} frames in {elapsed:.2f}s => {proc_fps:.2f} FPS")
# cap.release()

# df = pd.DataFrame(results_rows, columns=[
#     "frame_idx", "track_id", "x1", "y1", "x2", "y2",
#     "confirmed", "actions", "scores"
# ])
# df = df.drop_duplicates(subset=["frame_idx", "track_id"], keep="last")
# df.to_csv(CONFIG["RESULTS_CSV"], index=False)
# print(f"[INFO] Saved tracking + actions CSV → {CONFIG['RESULTS_CSV']}")


In [14]:
# import cv2
# import numpy as np
# import statistics
# import torch

# # ============================================================
# # 🎯 AUTO PARAMETER ADVISOR
# # ============================================================
# def auto_tune_parameters(video_path, yolo_model_path, sample_frames=100):
#     """
#     Analyzes a video to recommend YOLO + tracker + action classifier parameters.
#     """

#     print(f"[INFO] Analyzing first {sample_frames} frames from: {os.path.basename(video_path)}")

#     cap = cv2.VideoCapture(video_path)
#     if not cap.isOpened():
#         raise RuntimeError(f"Cannot open video: {video_path}")

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     print(f"[INFO] Resolution: {w}x{h}, FPS: {fps:.2f}, Total Frames: {total_frames}")

#     from ultralytics import YOLO
#     yolo = YOLO(yolo_model_path)
#     conf_candidates = [0.25, 0.5, 0.7]

#     frame_indices = np.linspace(0, min(sample_frames, total_frames - 1), min(sample_frames, total_frames), dtype=int)
#     motion_scores, object_counts = [], []
#     last_gray = None

#     for idx in frame_indices:
#         cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#         ret, frame = cap.read()
#         if not ret:
#             continue

#         # Compute motion (frame difference)
#         gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#         if last_gray is not None:
#             diff = cv2.absdiff(gray, last_gray)
#             motion_scores.append(np.mean(diff))
#         last_gray = gray

#         # Run quick detection
#         res = yolo.predict(frame, conf=0.5, verbose=False)
#         count = len(res[0].boxes)
#         object_counts.append(count)

#     cap.release()

#     avg_motion = np.mean(motion_scores)
#     motion_std = np.std(motion_scores)
#     avg_objects = np.mean(object_counts)
#     obj_std = np.std(object_counts)

#     print(f"\n[INFO] Scene motion intensity: {avg_motion:.2f} ± {motion_std:.2f}")
#     print(f"[INFO] Avg detected objects/frame: {avg_objects:.2f} ± {obj_std:.2f}")

#     # ------------------------------------------------------------
#     # 🧠 Rule-based suggestions
#     # ------------------------------------------------------------
#     # Confidence threshold
#     if avg_objects > 20:
#         conf = 0.6  # crowded
#     elif avg_objects > 5:
#         conf = 0.5
#     else:
#         conf = 0.7  # sparse scenes prefer higher confidence

#     # Detection frequency
#     if avg_motion > 12:
#         detect_every = 1  # high motion → frequent detection
#     elif avg_motion > 6:
#         detect_every = 3
#     else:
#         detect_every = 5  # low motion → rely more on tracker

#     # Out-of-frame fraction
#     out_frac = 0.8 if avg_motion > 8 else 0.9

#     # Re-ID memory
#     reid_time = 2.0 if avg_motion > 10 else 4.0

#     # TimeSformer clip length
#     if fps >= 30:
#         num_frames = 8
#     elif fps >= 20:
#         num_frames = 12
#     else:
#         num_frames = 16

#     # Top-K actions
#     top_k = 2 if num_frames <= 12 else 3

#     # ------------------------------------------------------------
#     print("\n⚙️  Recommended Parameters")
#     print("="*50)
#     print(f"CONF_THRESH           = {conf}")
#     print(f"DETECT_EVERY_N_FRAMES = {detect_every}")
#     print(f"OUT_OF_FRAME_FRACTION = {out_frac}")
#     print(f"REID_MEMORY_TIME      = {reid_time}")
#     print(f"NUM_FRAMES (TimeSformer clip) = {num_frames}")
#     print(f"TOP_K (actions)       = {top_k}")
#     print("="*50)

#     suggested = {
#         "CONF_THRESH": conf,
#         "DETECT_EVERY_N_FRAMES": detect_every,
#         "OUT_OF_FRAME_FRACTION": out_frac,
#         "REID_MEMORY_TIME": reid_time,
#         "NUM_FRAMES": num_frames,
#         "TOP_K": top_k
#     }
#     return suggested


# # ============================================================
# # 🧪 Run on your current video
# # ============================================================
# suggested_params = auto_tune_parameters(CONFIG["VIDEO_PATH"], CONFIG["YOLO_MODEL_PATH"])

# # You can update CONFIG automatically:
# CONFIG.update(suggested_params)
# print("\n[✅ CONFIG UPDATED WITH RECOMMENDED VALUES]")
